In [7]:
files = [      "prod_C11143/prod_C11143_2025-10-04-21-01-59_22.bag",
    "prod_C11143/prod_C11143_2025-10-04-21-10-29_h264.mcap",
    "prod_C11143/prod_C11143_2025-10-04-21-11-59_23.bag",
    "prod_C11143/prod_C11143_2025-10-04-21-12-31_h264.mcap",
    "prod_C11143/prod_C11143_2025-10-04-21-14-33_h264.mcap",
    "prod_C11143/prod_C11143_2025-10-04-21-20-39_h264.mcap"]

In [ ]:
from dataclasses import dataclass
from typing import Iterable, Optional
import re
from datetime import datetime

_TS_RE = re.compile(r"(\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2})")

@dataclass(frozen=True)
class Interval:
    start: float
    end: float

def merge_intervals(intervals: Iterable[Interval], gap_tol_s: float = 0.0) -> list[Interval]:
    xs = sorted(intervals, key=lambda i: i.start)
    if not xs:
        return []
    out = [xs[0]]
    for cur in xs[1:]:
        last = out[-1]
        if cur.start <= last.end + gap_tol_s:  # merge if overlapping or "close enough"
            out[-1] = Interval(last.start, max(last.end, cur.end))
        else:
            out.append(cur)
    return out

def intersect_intervals(a: list[Interval], b: list[Interval]) -> list[Interval]:
    i = j = 0
    out: list[Interval] = []
    while i < len(a) and j < len(b):
        s = max(a[i].start, b[j].start)
        e = min(a[i].end, b[j].end)
        if s < e:
            out.append(Interval(s, e))
        if a[i].end < b[j].end:
            i += 1
        else:
            j += 1
    return out

def largest_contiguous_overlap(
    bag_intervals: Iterable[tuple[float, float]],
    video_intervals: Iterable[tuple[float, float]],
    *,
    gap_tol_s: float = 3.0,
) -> Optional[Interval]:
    bags = merge_intervals((Interval(s, e) for s, e in bag_intervals), gap_tol_s=gap_tol_s)
    vids = merge_intervals((Interval(s, e) for s, e in video_intervals), gap_tol_s=gap_tol_s)

    overlaps = intersect_intervals(bags, vids)
    overlaps = merge_intervals(overlaps, gap_tol_s=gap_tol_s)  # "contiguous overlap" with tolerated gaps

    return max(overlaps, key=lambda iv: iv.end - iv.start, default=None)



def parse_timestamp_from_mcap(path: str) -> datetime:
    m = _TS_RE.search(path)
    if not m:
        raise ValueError(f"no timestamp found in {path}")
    return datetime.strptime(m.group(1), "%Y-%m-%d-%H-%M-%S").timestamp()

def find_valid_logs(files):
  bags = [f for f in files if f.endswith(".bag")]
  bag_starts = [parse_timestamp_from_mcap(f) for f in bags]
  bag_intervals_iv = [(s, s + 10 * 60) for s in bag_starts]

  videos = [f for f in files if f.endswith("_h264.mcap")]
  video_starts = [parse_timestamp_from_mcap(f) for f in videos]
  video_intervals_iv = [(s, s + 1 * 60) for s in video_starts]

  # TODO(Brad): find largest contiguous overlap
  best = largest_contiguous_overlap(bag_intervals_iv, video_intervals_iv, gap_tol_s=5.0)
  return best

In [9]:
import re
from datetime import datetime

_TS_RE = re.compile(r"(\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2})")

def parse_timestamp_from_mcap(path: str) -> datetime:
    m = _TS_RE.search(path)
    if not m:
        raise ValueError(f"no timestamp found in {path}")
    return datetime.strptime(m.group(1), "%Y-%m-%d-%H-%M-%S").timestamp()

bags = [f for f in files if f.endswith(".bag")]
bag_starts = [parse_timestamp_from_mcap(f) for f in bags]
bag_intervals_iv = [(s, s + 10 * 60) for s in bag_starts]

videos = [f for f in files if f.endswith("_h264.mcap")]
video_starts = [parse_timestamp_from_mcap(f) for f in videos]
video_intervals_iv = [(s, s + 1 * 60) for s in video_starts]

# TODO(Brad): find largest contiguous overlap
best = largest_contiguous_overlap(bag_intervals_iv, video_intervals_iv, gap_tol_s=5.0)
if best is None:
    print("No overlap")
else:
    print("Best overlap seconds:", best.end - best.start)
    print("Overlap window:", best.start, best.end)

Best overlap seconds: 60.0
Overlap window: 1759637429.0 1759637489.0
